# Stability-Guided Ensemble Pruning for Improved Generalization

This notebook implements a stability-guided ensemble pruning pipeline that:
1. Evaluates individual classifier stability via repeated cross-validation
2. Prunes unstable/weak models based on ROC-AUC mean and variance
3. Compares pruned ensembles against full ensembles and individual models
4. Tunes only the pruned subset for final performance

**Datasets:** Heart Disease (UCI) and Pima Indians Diabetes

In [12]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import (
    make_scorer, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    matthews_corrcoef, confusion_matrix
)
from xgboost import XGBClassifier

from ucimlrepo import fetch_ucirepo

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Output directories ---
DATASETS_DIR = 'datasets'
PLOTS_DIR = 'plots'
os.makedirs(DATASETS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# --- Custom scorers ---
def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0

SCORING = {
    'accuracy': 'accuracy',
    'recall': 'recall',
    'precision': 'precision',
    'specificity': make_scorer(specificity_score),
    'f1_weighted': 'f1_weighted',
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',
    'mcc': make_scorer(matthews_corrcoef)
}

print('All imports successful.')
print(f'Output directories: {DATASETS_DIR}/, {PLOTS_DIR}/')

All imports successful.
Output directories: datasets/, plots/


## 1. Load Datasets

In [13]:
# --- Heart Disease (UCI) ---
heart_repo = fetch_ucirepo(id=45)
heart_X = heart_repo.data.features.copy()
heart_y = heart_repo.data.targets.copy()

# The target have values 0-4; binarise: 0 = no disease, 1 = disease
heart_y.columns = ['target']
heart_y['target'] = (heart_y['target'] > 0).astype(int)

heart_df = pd.concat([heart_X, heart_y], axis=1)
heart_df.to_csv(f'{DATASETS_DIR}/heart_disease_clean.csv', index=False)
print(f'Heart Disease shape: {heart_df.shape}')
print(f'Class distribution:\n{heart_df["target"].value_counts()}')
print(f'Features: {list(heart_X.columns)}')
print()

Heart Disease shape: (303, 14)
Class distribution:
target
0    164
1    139
Name: count, dtype: int64
Features: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']



In [14]:
# --- Pima Indians Diabetes ---
pima_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
             'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
pima_df = pd.read_csv(
    'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv',
    header=None, names=pima_cols
)
pima_df.to_csv(f'{DATASETS_DIR}/pima_diabetes_clean.csv', index=False)
print(f'Pima Diabetes shape: {pima_df.shape}')
print(f'Class distribution:\n{pima_df["Outcome"].value_counts()}')
print(f'Features: {pima_cols[:-1]}')
print()

Pima Diabetes shape: (768, 9)
Class distribution:
Outcome
0    500
1    268
Name: count, dtype: int64
Features: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']



## 2. Preprocessing

- **Heart Disease**: `SimpleImputer(median)` + `StandardScaler` via sklearn `Pipeline`
- **Pima Diabetes**: `IterativeImputer` + `StandardScaler` + `SMOTE` via imblearn `Pipeline`

All preprocessing is performed **inside each CV fold** to prevent data leakage.
SMOTE is applied only to training folds via the imblearn pipeline.

In [15]:
# Prepare X, y for each dataset
# Heart Disease
heart_X = heart_df.drop('target', axis=1)
heart_y = heart_df['target'].values

# Replace any zero values in columns where 0 is physiologically impossible with NaN
# (Heart dataset may have missing values encoded as NaN already from UCI)
print('Heart Disease missing values per column:')
print(heart_X.isnull().sum())
print()

# Pima Diabetes — zeros in Glucose, BloodPressure, SkinThickness, Insulin, BMI are missing
pima_X = pima_df.drop('Outcome', axis=1).copy()
pima_y = pima_df['Outcome'].values

zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
pima_X[zero_cols] = pima_X[zero_cols].replace(0, np.nan)
print('Pima Diabetes missing values per column (after replacing 0s):')
print(pima_X.isnull().sum())
print()

# Check class balance
for name, y in [('Heart', heart_y), ('Pima', pima_y)]:
    counts = np.bincount(y)
    ratio = counts.min() / counts.max()
    print(f'{name}: class 0 = {counts[0]}, class 1 = {counts[1]}, minority ratio = {ratio:.2f}')
    if ratio < 0.5:
        print(f'  -> Imbalanced! Will include PR-AUC in evaluation.')
    else:
        print(f'  -> Reasonably balanced.')
print()

Heart Disease missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
dtype: int64

Pima Diabetes missing values per column (after replacing 0s):
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
dtype: int64

Heart: class 0 = 164, class 1 = 139, minority ratio = 0.85
  -> Reasonably balanced.
Pima: class 0 = 500, class 1 = 268, minority ratio = 0.54
  -> Reasonably balanced.



## 3. Define Classifier Pool (Default Hyperparameters)

All classifiers use **default** hyperparameters. This is intentional to ensure a fair, unbiased,
leakage-free stability comparison across all models.

In [16]:
def get_base_classifiers():
    """Return dict of name -> classifier with default hyperparameters."""
    return {
        'LR': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
        'SVM': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
        'kNN': KNeighborsClassifier(n_neighbors=5),
        'RF': RandomForestClassifier(random_state=RANDOM_STATE),
        'XGBoost': XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss',
                                  use_label_encoder=False)
    }

def make_heart_pipeline(clf):
    """Heart Disease pipeline: SimpleImputer + StandardScaler."""
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', clf)
    ])

def make_pima_pipeline(clf):
    """Pima Diabetes pipeline: IterativeImputer + StandardScaler + SMOTE."""
    return ImbPipeline([
        ('imputer', IterativeImputer(max_iter=10, random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('clf', clf)
    ])

def make_pipeline(clf, dataset_name):
    """Dataset-aware pipeline builder."""
    if dataset_name == 'pima':
        return make_pima_pipeline(clf)
    return make_heart_pipeline(clf)

def make_heart_ensemble_pipeline(ensemble):
    """Heart Disease ensemble pipeline."""
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('ensemble', ensemble)
    ])

def make_pima_ensemble_pipeline(ensemble):
    """Pima Diabetes ensemble pipeline: IterativeImputer + Scaler + SMOTE + ensemble."""
    return ImbPipeline([
        ('imputer', IterativeImputer(max_iter=10, random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('ensemble', ensemble)
    ])

def make_ensemble_pipeline(ensemble, dataset_name):
    """Dataset-aware ensemble pipeline builder."""
    if dataset_name == 'pima':
        return make_pima_ensemble_pipeline(ensemble)
    return make_heart_ensemble_pipeline(ensemble)

print('Classifier pool defined: LR, SVM, kNN, RF, XGBoost')
print('Heart pipeline: SimpleImputer(median) -> StandardScaler -> clf')
print('Pima pipeline:  IterativeImputer -> StandardScaler -> SMOTE -> clf')

Classifier pool defined: LR, SVM, kNN, RF, XGBoost
Heart pipeline: SimpleImputer(median) -> StandardScaler -> clf
Pima pipeline:  IterativeImputer -> StandardScaler -> SMOTE -> clf


## 4. Stage 1 — Stability Evaluation (Default Params)

RepeatedStratifiedKFold (5 splits x 5 repeats = 25 scores per model).  
Scaler is fit **only on training data** per fold via Pipeline.

In [17]:
def extract_metrics(cv_results):
    """Extract all metric arrays from cross_validate results."""
    return {
        'accuracy': cv_results['test_accuracy'],
        'recall': cv_results['test_recall'],
        'precision': cv_results['test_precision'],
        'specificity': cv_results['test_specificity'],
        'f1_weighted': cv_results['test_f1_weighted'],
        'roc_auc': cv_results['test_roc_auc'],
        'pr_auc': cv_results['test_pr_auc'],
        'mcc': cv_results['test_mcc'],
    }

def metrics_to_row(name_key, name_val, metrics):
    """Convert metric arrays to a summary row dict."""
    auc = metrics['roc_auc']
    stability = auc.mean() / (auc.std() + 1e-9)
    row = {name_key: name_val}
    for metric_name, arr in metrics.items():
        nice = (metric_name.replace('_', ' ').title()
                .replace('Roc Auc', 'AUC').replace('Pr Auc', 'PR-AUC')
                .replace('F1 Weighted', 'F1').replace('Mcc', 'MCC'))
        row[f'Mean {nice}'] = arr.mean()
        row[f'Std {nice}'] = arr.std()
    row['Stability Score'] = stability
    return row

DISPLAY_COLS = ['Mean Accuracy', 'Mean Recall', 'Mean Precision', 'Mean Specificity',
                'Mean F1', 'Mean AUC', 'Mean PR-AUC', 'Mean MCC', 'Std AUC', 'Stability Score']

def evaluate_stability(X, y, dataset_name):
    """Run Stage 1 stability evaluation with expanded metrics."""
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    classifiers = get_base_classifiers()
    results = []
    fold_scores = {}

    for name, clf in classifiers.items():
        pipe = make_pipeline(clf, dataset_name)
        cv_results = cross_validate(pipe, X, y, cv=rskf, scoring=SCORING,
                                     return_train_score=False, n_jobs=-1)
        metrics = extract_metrics(cv_results)
        fold_scores[name] = metrics['roc_auc']
        results.append(metrics_to_row('Model', name, metrics))

    df = pd.DataFrame(results)
    df.to_csv(f'{DATASETS_DIR}/stability_results_{dataset_name}.csv', index=False)

    print(f'\n=== Stability Results — {dataset_name.upper()} ===')
    display_df = df[['Model'] + DISPLAY_COLS].copy()
    for col in DISPLAY_COLS:
        display_df[col] = df[col].map(lambda x: f'{x:.4f}')
    print(display_df.to_string(index=False))
    return df, fold_scores

print('Running stability evaluation for Heart Disease...')
heart_stability_df, heart_fold_scores = evaluate_stability(heart_X, heart_y, 'heart')

print('\nRunning stability evaluation for Pima Diabetes...')
pima_stability_df, pima_fold_scores = evaluate_stability(pima_X, pima_y, 'pima')

Running stability evaluation for Heart Disease...

=== Stability Results — HEART ===
  Model Mean Accuracy Mean Recall Mean Precision Mean Specificity Mean F1 Mean AUC Mean PR-AUC Mean MCC Std AUC Stability Score
     LR        0.8375      0.7914         0.8506           0.8769  0.8365   0.9059      0.9040   0.6766  0.0391         23.1836
    SVM        0.8342      0.7783         0.8521           0.8818  0.8331   0.8990      0.8973   0.6690  0.0378         23.7678
    kNN        0.8303      0.7885         0.8344           0.8659  0.8295   0.8907      0.8578   0.6596  0.0387         23.0077
     RF        0.8315      0.7769         0.8486           0.8780  0.8302   0.9047      0.8985   0.6644  0.0369         24.4944
XGBoost        0.8038      0.7639         0.8034           0.8377  0.8027   0.8787      0.8741   0.6076  0.0358         24.5689

Running stability evaluation for Pima Diabetes...

=== Stability Results — PIMA ===
  Model Mean Accuracy Mean Recall Mean Precision Mean Specific

## 5. Stage 2 — Pruning

Select models where **both**:
- Mean ROC-AUC >= average of all models' Mean AUC
- Std ROC-AUC <= average of all models' Std AUC

If fewer than 2 models selected, relax threshold slightly and retry.

In [18]:
def prune_models(stability_df, dataset_name):
    """Select stable, high-performing models. Returns list of selected model names."""
    mean_auc_threshold = stability_df['Mean AUC'].mean()
    std_auc_threshold = stability_df['Std AUC'].mean()
    
    relaxation = 0.0
    selected = []
    
    while len(selected) < 2:
        auc_thresh = mean_auc_threshold - relaxation
        std_thresh = std_auc_threshold + relaxation
        
        mask = (stability_df['Mean AUC'] >= auc_thresh) & (stability_df['Std AUC'] <= std_thresh)
        selected = stability_df[mask]['Model'].tolist()
        
        if len(selected) < 2:
            relaxation += 0.005
            if relaxation > 0.1:
                # Fallback: pick top 2 by stability score
                selected = stability_df.nlargest(2, 'Stability Score')['Model'].tolist()
                break
    
    print(f'\n=== Pruning Results — {dataset_name.upper()} ===')
    print(f'Mean AUC threshold: {auc_thresh:.4f} (original: {mean_auc_threshold:.4f})')
    print(f'Std AUC threshold:  {std_thresh:.4f} (original: {std_auc_threshold:.4f})')
    if relaxation > 0:
        print(f'(Threshold relaxed by {relaxation:.3f} to ensure >= 2 models selected)')
    print()
    
    for _, row in stability_df.iterrows():
        name = row['Model']
        if name in selected:
            print(f'  SELECTED: {name} — Mean AUC={row["Mean AUC"]:.4f}, Std AUC={row["Std AUC"]:.4f}')
        else:
            reasons = []
            if row['Mean AUC'] < auc_thresh:
                reasons.append(f'Mean AUC {row["Mean AUC"]:.4f} < {auc_thresh:.4f}')
            if row['Std AUC'] > std_thresh:
                reasons.append(f'Std AUC {row["Std AUC"]:.4f} > {std_thresh:.4f}')
            print(f'  REJECTED: {name} — {"; ".join(reasons)}')
    
    return selected

heart_selected = prune_models(heart_stability_df, 'heart')
pima_selected = prune_models(pima_stability_df, 'pima')


=== Pruning Results — HEART ===
Mean AUC threshold: 0.8908 (original: 0.8958)
Std AUC threshold:  0.0427 (original: 0.0377)
(Threshold relaxed by 0.005 to ensure >= 2 models selected)

  SELECTED: LR — Mean AUC=0.9059, Std AUC=0.0391
  SELECTED: SVM — Mean AUC=0.8990, Std AUC=0.0378
  REJECTED: kNN — Mean AUC 0.8907 < 0.8908
  SELECTED: RF — Mean AUC=0.9047, Std AUC=0.0369
  REJECTED: XGBoost — Mean AUC 0.8787 < 0.8908

=== Pruning Results — PIMA ===
Mean AUC threshold: 0.8138 (original: 0.8138)
Std AUC threshold:  0.0325 (original: 0.0325)

  SELECTED: LR — Mean AUC=0.8366, Std AUC=0.0310
  SELECTED: SVM — Mean AUC=0.8258, Std AUC=0.0290
  REJECTED: kNN — Mean AUC 0.7802 < 0.8138; Std AUC 0.0394 > 0.0325
  SELECTED: RF — Mean AUC=0.8258, Std AUC=0.0324
  REJECTED: XGBoost — Mean AUC 0.8008 < 0.8138


## 6. Stage 3 — Build & Evaluate All Methods

Evaluate 4 method categories (individuals already done in Stage 1):
1. Individual classifiers (reuse Stage 1)
2. Full Soft Voting Ensemble
3. Full Stacking Ensemble
4. Pruned Soft Voting Ensemble (default params)

In [19]:
def evaluate_ensemble_metrics(X, y, ensemble_pipe, rskf):
    """Evaluate an ensemble pipeline and return metric arrays."""
    cv_results = cross_validate(ensemble_pipe, X, y, cv=rskf, scoring=SCORING,
                                 return_train_score=False, n_jobs=-1)
    return extract_metrics(cv_results)


def build_and_evaluate_ensembles(X, y, selected_models, stability_df, fold_scores, dataset_name):
    """Build all ensemble methods and evaluate them."""
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    all_classifiers = get_base_classifiers()

    all_fold_scores = dict(fold_scores)
    results = []

    # 1. Individual classifiers — reuse from stability_df (convert Model -> Method)
    for _, row in stability_df.iterrows():
        r = dict(row)
        r['Method'] = f"{r.pop('Model')} (default)"
        results.append(r)

    # 2. Full Soft Voting Ensemble
    print(f'  Evaluating Full Soft Voting...')
    full_estimators = [(name, clf) for name, clf in all_classifiers.items()]
    full_voting_pipe = make_ensemble_pipeline(
        VotingClassifier(estimators=full_estimators, voting='soft'), dataset_name)
    metrics = evaluate_ensemble_metrics(X, y, full_voting_pipe, rskf)
    all_fold_scores['Full Soft Voting'] = metrics['roc_auc']
    results.append(metrics_to_row('Method', 'Full Soft Voting', metrics))

    # 3. Full Stacking Ensemble
    print(f'  Evaluating Full Stacking...')
    full_stacking_pipe = make_ensemble_pipeline(
        StackingClassifier(
            estimators=full_estimators,
            final_estimator=LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
            cv=5
        ), dataset_name)
    metrics = evaluate_ensemble_metrics(X, y, full_stacking_pipe, rskf)
    all_fold_scores['Full Stacking'] = metrics['roc_auc']
    results.append(metrics_to_row('Method', 'Full Stacking', metrics))

    # 4. Pruned Soft Voting Ensemble (default params)
    print(f'  Evaluating Pruned Soft Voting (default)...')
    pruned_estimators = [(name, all_classifiers[name]) for name in selected_models]
    pruned_voting_pipe = make_ensemble_pipeline(
        VotingClassifier(estimators=pruned_estimators, voting='soft'), dataset_name)
    metrics = evaluate_ensemble_metrics(X, y, pruned_voting_pipe, rskf)
    all_fold_scores['Pruned Soft Voting (default)'] = metrics['roc_auc']
    results.append(metrics_to_row('Method', 'Pruned Soft Voting (default)', metrics))

    return results, all_fold_scores


print('=== Stage 3 — Heart Disease ===')
heart_results, heart_all_fold_scores = build_and_evaluate_ensembles(
    heart_X, heart_y, heart_selected, heart_stability_df, heart_fold_scores, 'heart')

print('\n=== Stage 3 — Pima Diabetes ===')
pima_results, pima_all_fold_scores = build_and_evaluate_ensembles(
    pima_X, pima_y, pima_selected, pima_stability_df, pima_fold_scores, 'pima')

=== Stage 3 — Heart Disease ===
  Evaluating Full Soft Voting...
  Evaluating Full Stacking...
  Evaluating Pruned Soft Voting (default)...

=== Stage 3 — Pima Diabetes ===
  Evaluating Full Soft Voting...
  Evaluating Full Stacking...
  Evaluating Pruned Soft Voting (default)...


## 7. Stage 4 — Hyperparameter Tuning (Pruned Models Only)

Tuning happens **after** pruning. Pruning was done on neutral default params to avoid bias.  
Tuning here serves only to enhance the final pruned ensemble.

In [20]:
PARAM_GRIDS = {
    'LR': {'clf__C': [0.01, 0.1, 1, 10]},
    'SVM': {'clf__C': [0.1, 1, 10], 'clf__gamma': ['scale', 'auto']},
    'kNN': {'clf__n_neighbors': [3, 5, 7, 11]},
    'RF': {'clf__n_estimators': [100, 200], 'clf__max_depth': [None, 5, 10]},
    'XGBoost': {'clf__n_estimators': [100, 200], 'clf__learning_rate': [0.05, 0.1]}
}


def tune_and_build_pruned_ensemble(X, y, selected_models, dataset_name):
    """Tune each selected model via GridSearchCV, then build tuned pruned ensemble."""
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    all_classifiers = get_base_classifiers()

    tuned_classifiers = {}
    print(f'\n=== Hyperparameter Tuning — {dataset_name.upper()} ===')

    for name in selected_models:
        clf = all_classifiers[name]
        pipe = make_pipeline(clf, dataset_name)
        grid = PARAM_GRIDS[name]

        gs = GridSearchCV(pipe, grid, cv=3, scoring='roc_auc', n_jobs=-1)
        gs.fit(X, y)

        best_clf = gs.best_estimator_.named_steps['clf']
        tuned_classifiers[name] = best_clf
        print(f'  {name}: best params = {gs.best_params_}, best AUC = {gs.best_score_:.4f}')

    # Build tuned pruned voting ensemble
    tuned_estimators = [(name, tuned_classifiers[name]) for name in selected_models]
    tuned_pruned_pipe = make_ensemble_pipeline(
        VotingClassifier(estimators=tuned_estimators, voting='soft'), dataset_name)

    # Evaluate with expanded metrics
    print(f'  Evaluating Pruned Soft Voting (tuned)...')
    cv_results = cross_validate(tuned_pruned_pipe, X, y, cv=rskf, scoring=SCORING,
                                 return_train_score=False, n_jobs=-1)
    metrics = extract_metrics(cv_results)
    result = metrics_to_row('Method', 'Pruned Soft Voting (tuned)', metrics)

    return result, metrics['roc_auc']


heart_tuned_result, heart_tuned_auc = tune_and_build_pruned_ensemble(
    heart_X, heart_y, heart_selected, 'heart')

pima_tuned_result, pima_tuned_auc = tune_and_build_pruned_ensemble(
    pima_X, pima_y, pima_selected, 'pima')


=== Hyperparameter Tuning — HEART ===
  LR: best params = {'clf__C': 0.01}, best AUC = 0.9046
  SVM: best params = {'clf__C': 0.1, 'clf__gamma': 'scale'}, best AUC = 0.9001
  RF: best params = {'clf__max_depth': 5, 'clf__n_estimators': 200}, best AUC = 0.9059
  Evaluating Pruned Soft Voting (tuned)...

=== Hyperparameter Tuning — PIMA ===
  LR: best params = {'clf__C': 0.1}, best AUC = 0.8367
  SVM: best params = {'clf__C': 0.1, 'clf__gamma': 'scale'}, best AUC = 0.8237
  RF: best params = {'clf__max_depth': 5, 'clf__n_estimators': 200}, best AUC = 0.8369
  Evaluating Pruned Soft Voting (tuned)...


## 8. Final Results Table

In [21]:
FINAL_DISPLAY_COLS = ['Mean Accuracy', 'Mean Recall', 'Mean Precision', 'Mean Specificity',
                      'Mean F1', 'Mean AUC', 'Mean PR-AUC', 'Mean MCC',
                      'Std Accuracy', 'Std Recall', 'Std Precision', 'Std Specificity',
                      'Std F1', 'Std AUC', 'Std PR-AUC', 'Std MCC']

def build_final_table(results_list, tuned_result, dataset_name):
    """Combine all results into master DataFrame, highlight best values."""
    all_results = results_list + [tuned_result]
    df = pd.DataFrame(all_results)
    df = df.set_index('Method')

    cols_to_save = [c for c in FINAL_DISPLAY_COLS if c in df.columns and not df[c].isna().all()]
    df = df[cols_to_save]
    df.to_csv(f'{DATASETS_DIR}/final_results_{dataset_name}.csv')

    print(f'\n{"=" * 100}')
    print(f'FINAL RESULTS — {dataset_name.upper()}')
    print(f'{"=" * 100}')

    compact_cols = ['Mean Accuracy', 'Mean Recall', 'Mean Precision', 'Mean Specificity',
                    'Mean F1', 'Mean AUC', 'Mean PR-AUC', 'Mean MCC', 'Std AUC']
    compact_cols = [c for c in compact_cols if c in df.columns and not df[c].isna().all()]
    display_df = df[compact_cols].copy()

    for col in compact_cols:
        display_df[col] = df[col].map(lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')

    for col in compact_cols:
        col_data = df[col].dropna()
        if col_data.empty:
            continue
        best_idx = col_data.idxmin() if 'Std' in col else col_data.idxmax()
        display_df.loc[best_idx, col] = display_df.loc[best_idx, col] + ' *'

    print(display_df.to_string())
    print('\n(* = best value in column)')
    return df

heart_final_df = build_final_table(heart_results, heart_tuned_result, 'heart')
pima_final_df = build_final_table(pima_results, pima_tuned_result, 'pima')


FINAL RESULTS — HEART
                             Mean Accuracy Mean Recall Mean Precision Mean Specificity   Mean F1  Mean AUC Mean PR-AUC  Mean MCC   Std AUC
Method                                                                                                                                    
LR (default)                        0.8375    0.7914 *         0.8506           0.8769    0.8365    0.9059      0.9040    0.6766    0.0391
SVM (default)                       0.8342      0.7783         0.8521           0.8818    0.8331    0.8990      0.8973    0.6690    0.0378
kNN (default)                       0.8303      0.7885         0.8344           0.8659    0.8295    0.8907      0.8578    0.6596    0.0387
RF (default)                        0.8315      0.7769         0.8486           0.8780    0.8302    0.9047      0.8985    0.6644    0.0369
XGBoost (default)                   0.8038      0.7639         0.8034           0.8377    0.8027    0.8787      0.8741    0.6076    0.0358
Full

## 9. Visualizations

In [22]:
# --- Plot 1: Bar chart — ROC-AUC mean + error bars ---
def plot_auc_comparison(final_df, dataset_name):
    fig, ax = plt.subplots(figsize=(12, 6))
    methods = final_df.index.tolist()
    means = final_df['Mean AUC'].values
    stds = final_df['Std AUC'].values

    colors = sns.color_palette('viridis', len(methods))
    ax.bar(range(len(methods)), means, yerr=stds, capsize=4,
           color=colors, edgecolor='black', linewidth=0.5)

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('ROC-AUC')
    ax.set_title(f'ROC-AUC Comparison — {dataset_name.upper()}')
    ax.set_ylim(0.5, 1.0)
    plt.tight_layout()
    path = f'{PLOTS_DIR}/plot_auc_comparison_{dataset_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {path}')

plot_auc_comparison(heart_final_df, 'heart')
plot_auc_comparison(pima_final_df, 'pima')

Saved plots/plot_auc_comparison_heart.png
Saved plots/plot_auc_comparison_pima.png


In [23]:
# --- Plot 2: Heatmap — stability scores across both datasets ---
model_names = ['LR', 'SVM', 'kNN', 'RF', 'XGBoost']
heart_stab = heart_stability_df.set_index('Model')['Stability Score']
pima_stab = pima_stability_df.set_index('Model')['Stability Score']

heatmap_df = pd.DataFrame({
    'Heart Disease': [heart_stab[m] for m in model_names],
    'Pima Diabetes': [pima_stab[m] for m in model_names]
}, index=model_names)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='gray')
ax.set_title('Stability Scores (Mean AUC / Std AUC) Across Datasets')
ax.set_ylabel('Model')
plt.tight_layout()
path = f'{PLOTS_DIR}/plot_stability_heatmap.png'
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved {path}')

Saved plots/plot_stability_heatmap.png


In [24]:
# --- Plot 3: Box plots — AUC distribution across folds ---
def plot_boxplot(fold_scores, tuned_auc, dataset_name):
    all_scores = dict(fold_scores)
    all_scores['Pruned Soft Voting (tuned)'] = tuned_auc

    order = ['LR', 'SVM', 'kNN', 'RF', 'XGBoost',
             'Full Soft Voting', 'Full Stacking',
             'Pruned Soft Voting (default)', 'Pruned Soft Voting (tuned)']

    data, labels = [], []
    for name in order:
        if name in all_scores:
            data.append(all_scores[name])
            labels.append(name)

    fig, ax = plt.subplots(figsize=(14, 6))
    bp = ax.boxplot(data, patch_artist=True, notch=True)

    colors = sns.color_palette('viridis', len(labels))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('ROC-AUC')
    ax.set_title(f'ROC-AUC Distribution Across Folds — {dataset_name.upper()}')
    plt.tight_layout()
    path = f'{PLOTS_DIR}/plot_boxplot_{dataset_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {path}')

plot_boxplot(heart_all_fold_scores, heart_tuned_auc, 'heart')
plot_boxplot(pima_all_fold_scores, pima_tuned_auc, 'pima')

Saved plots/plot_boxplot_heart.png
Saved plots/plot_boxplot_pima.png


In [25]:
# --- Plot 4: Grouped bar chart — Recall vs Specificity ---
def plot_recall_specificity(final_df, dataset_name):
    methods = final_df.index.tolist()
    recall = final_df['Mean Recall'].values
    spec = final_df['Mean Specificity'].values

    x = np.arange(len(methods))
    width = 0.35

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(x - width/2, recall, width, label='Recall (Sensitivity)',
           color=sns.color_palette('viridis', 2)[0], edgecolor='black', linewidth=0.5)
    ax.bar(x + width/2, spec, width, label='Specificity',
           color=sns.color_palette('viridis', 2)[1], edgecolor='black', linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Score')
    ax.set_title(f'Recall vs Specificity — {dataset_name.upper()}')
    ax.set_ylim(0.0, 1.05)
    ax.legend()
    plt.tight_layout()
    path = f'{PLOTS_DIR}/plot_recall_spec_{dataset_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {path}')

plot_recall_specificity(heart_final_df, 'heart')
plot_recall_specificity(pima_final_df, 'pima')

Saved plots/plot_recall_spec_heart.png
Saved plots/plot_recall_spec_pima.png


In [26]:
# --- Plot 5: MCC comparison bar chart ---
def plot_mcc(final_df, dataset_name):
    methods = final_df.index.tolist()
    mcc_vals = final_df['Mean MCC'].values

    fig, ax = plt.subplots(figsize=(12, 6))
    colors = sns.color_palette('viridis', len(methods))
    ax.bar(range(len(methods)), mcc_vals, color=colors, edgecolor='black', linewidth=0.5)

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('MCC')
    ax.set_title(f'Matthews Correlation Coefficient — {dataset_name.upper()}')
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    path = f'{PLOTS_DIR}/plot_mcc_{dataset_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {path}')

plot_mcc(heart_final_df, 'heart')
plot_mcc(pima_final_df, 'pima')

Saved plots/plot_mcc_heart.png
Saved plots/plot_mcc_pima.png


## 10. Output Files Summary

In [27]:
import os

expected_files = [
    f'{DATASETS_DIR}/heart_disease_clean.csv',
    f'{DATASETS_DIR}/pima_diabetes_clean.csv',
    f'{DATASETS_DIR}/stability_results_heart.csv',
    f'{DATASETS_DIR}/stability_results_pima.csv',
    f'{DATASETS_DIR}/final_results_heart.csv',
    f'{DATASETS_DIR}/final_results_pima.csv',
    f'{PLOTS_DIR}/plot_auc_comparison_heart.png',
    f'{PLOTS_DIR}/plot_auc_comparison_pima.png',
    f'{PLOTS_DIR}/plot_stability_heatmap.png',
    f'{PLOTS_DIR}/plot_boxplot_heart.png',
    f'{PLOTS_DIR}/plot_boxplot_pima.png',
    f'{PLOTS_DIR}/plot_recall_spec_heart.png',
    f'{PLOTS_DIR}/plot_recall_spec_pima.png',
    f'{PLOTS_DIR}/plot_mcc_heart.png',
    f'{PLOTS_DIR}/plot_mcc_pima.png',
]

print('=== Generated Files ===')
for f in expected_files:
    exists = os.path.isfile(f)
    size = os.path.getsize(f) if exists else 0
    status = f'{size:,} bytes' if exists else 'MISSING'
    print(f'  {f:55s} {status}')

print('\nDone! All stages completed successfully.')

=== Generated Files ===
  datasets/heart_disease_clean.csv                        12,475 bytes
  datasets/pima_diabetes_clean.csv                        23,290 bytes
  datasets/stability_results_heart.csv                    1,889 bytes
  datasets/stability_results_pima.csv                     1,875 bytes
  datasets/final_results_heart.csv                        3,165 bytes
  datasets/final_results_pima.csv                         3,135 bytes
  plots/plot_auc_comparison_heart.png                     66,591 bytes
  plots/plot_auc_comparison_pima.png                      66,677 bytes
  plots/plot_stability_heatmap.png                        47,062 bytes
  plots/plot_boxplot_heart.png                            100,727 bytes
  plots/plot_boxplot_pima.png                             98,782 bytes
  plots/plot_recall_spec_heart.png                        73,494 bytes
  plots/plot_recall_spec_pima.png                         74,094 bytes
  plots/plot_mcc_heart.png                              